## AI 모델 개발 15회차 1차 팀프로젝트 - NASA Turbofan Jet Engine Data Set 터보엔진 유지보전 문제_FD002

[데이터셋 개요]
이 데이터셋은 NASA의 C-MAPSS(Commercial Modular Aero-Propulsion System Simulation)
시뮬레이터를 사용하여 생성된 터보팬 엔진의 열화(Degradation) 데이터입니다.

데이터는 여러 개의 다변량 시계열(Multivariate Time Series)로 구성되어 있습니다.
데이터셋은 학습(Training) 세트와 테스트(Test) 세트로 나뉩니다.
각 시계열 데이터는 서로 다른 엔진(Unit)에서 얻은 데이터이며, 동일한 유형의 엔진들로 간주할 수 있습니다.

[실험 시나리오]
1. 초기 상태: 각 엔진은 정상 상태에서 작동을 시작합니다.
   - 단, 사용자에게는 알려지지 않은 수준의 초기 마모(Initial Wear)와 제조 공차(Variation)가 존재합니다.
     (이는 결함이 아닌 정상적인 범주입니다.)
2. 고장 진행: 어느 시점부터 결함(Fault)이 발생하여 시간이 지날수록 상태가 악화됩니다.
3. 데이터 범위:
   - 학습 세트 (Train): 결함 발생부터 시스템 고장(Failure) 시점까지의 모든 데이터가 포함됩니다.
   - 테스트 세트 (Test): 고장 발생 전 임의의 시점에서 데이터 기록이 중단됩니다.

[목표 (Objective)]
테스트 세트에 포함된 각 엔진의 **잔여 유효 수명(RUL: Remaining Useful Life)**을 예측하는 것입니다.
즉, 테스트 데이터가 끊긴 시점으로부터 엔진이 고장 날 때까지 몇 사이클(Cycle)이 더 남았는지 맞히는 것이 목표입니다.

> FD002 (Train: train_FD002.txt / Test: test_FD002.txt / RUL: RUL_FD002.txt)
   - 학습 엔진 수: 260개
   - 테스트 엔진 수: 259개
   - 운전 조건 (Conditions): 6가지 (다양한 고도, 속도, 부하 조건)
   - 고장 모드 (Fault Modes): 1가지 (HPC 열화)


> 여러 셀에서 사용되는 함수는 먼저 정의하고 시작한다. 코드 오류 방지

## 1. 라이브러리 및 데이터 로드

In [ ]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

RANDOM_STATE = 42

# ✅ 여기만 본인 환경에 맞게 바꾸세요.
DATA_DIR = "./"  # 예: Kaggle -> "/kaggle/input/nasa-cmaps/"
print("DATA_DIR:", os.path.abspath(DATA_DIR))
print("Files:", os.listdir(DATA_DIR)[:30])


## 2. 데이터 불러오기

In [ ]:
# 파일명은 데이터셋에 따라 다를 수 있어요. 목록을 보고 정확히 맞춰주세요.
TRAIN_FILE = r"C:\Users\yuzhd\Desktop\유진\data\CMaps\train_FD002.txt"
TEST_FILE  = r"C:\Users\yuzhd\Desktop\유진\data\CMaps\test_FD002.txt"
RUL_FILE   = r"C:\Users\yuzhd\Desktop\유진\data\CMaps\RUL_FD002.txt"

train_path = os.path.join(DATA_DIR, TRAIN_FILE)
test_path  = os.path.join(DATA_DIR, TEST_FILE)
rul_path   = os.path.join(DATA_DIR, RUL_FILE)

col_names = (
    ["unit_number", "time_in_cycles"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

def read_cmapss_txt(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None, names=col_names)
    # 어떤 파일은 마지막에 공백 컬럼이 생길 수 있어요(전부 NaN). 그런 컬럼 제거
    df = df.dropna(axis=1, how="all")
    return df

train_df = read_cmapss_txt(train_path)
test_df  = read_cmapss_txt(test_path)
rul_df = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["RUL"])


print("train:", train_df.shape, "test:", test_df.shape)
display(train_df.head())


In [ ]:
train_df.head()

## 2) Train RUL 만들기
Train은 각 엔진(unit)이 **고장날 때까지** 기록되어 있어서,
`(각 엔진의 마지막 사이클) - (현재 사이클)`로 RUL을 만들 수 있어요.

In [ ]:
max_cycle = train_df.groupby("unit_number")["time_in_cycles"].max().rename("max_cycle")

train_df = train_df.merge(max_cycle, on="unit_number", how="left")
train_df["RUL"] = train_df["max_cycle"] - train_df["time_in_cycles"]
train_df = train_df.drop(columns=["max_cycle"])

display(train_df[["unit_number","time_in_cycles","RUL"]].head(10))
print("RUL min/max:", train_df["RUL"].min(), train_df["RUL"].max())


## 3) Test 정답(RUL) 불러오기 & Test의 현재 RUL 만들기
Test는 각 엔진이 **고장 전 일부 구간만** 제공됩니다.
`RUL_FD001.txt`에는 각 test 엔진의 **마지막 관측 시점에서 남은 수명**이 1개씩 들어있어요.

따라서 test에서 각 row의 RUL은:
- `(engine_last_cycle - time_in_cycles) + RUL_at_last_observed`
로 계산합니다.

In [ ]:
import numpy as np
import pandas as pd

# test 엔진별 "마지막 관측 시점에서 남은 수명" (엔진별 1개씩)
rul_at_last = pd.read_csv(rul_path, header=None, names=["RUL_at_last"])
rul_at_last["unit_number"] = np.arange(1, len(rul_at_last) + 1)

# test 엔진별 마지막 관측 cycle
test_last_cycle = test_df.groupby("unit_number")["time_in_cycles"].max().rename("last_cycle")

test_df = test_df.merge(test_last_cycle, on="unit_number", how="left")
test_df = test_df.merge(rul_at_last, on="unit_number", how="left")

# 각 row의 RUL 계산
test_df["RUL"] = (test_df["last_cycle"] - test_df["time_in_cycles"]) + test_df["RUL_at_last"]

test_df = test_df.drop(columns=["last_cycle", "RUL_at_last"])

print("test_df shape:", test_df.shape)
display(test_df[["unit_number", "time_in_cycles", "RUL"]].head(10))
print("Test RUL min/max:", test_df["RUL"].min(), test_df["RUL"].max())


## 3. Feature Engineering

In [ ]:
rename_dict = {
    "op_setting_1": "Alt[kft]", # Altitude
    "op_setting_2": "Mn[-]", # Mach number
    "op_setting_3": "TLA[deg]", # Thrust lever angle (detent?)
    "sensor_1": "T2[R]",  # Total temperature at fan inlet 
    "sensor_2": "T24[R]", # Total temperature at LPC outlet
    "sensor_3": "T30[R]", # Total temperature at HPC outlet
    "sensor_4": "T50[R]", # Total temperature at LPT outlet
    "sensor_5": "P2[psi]", # Pressure at fan inlet
    "sensor_6": "P15[psi]", # Total pressure in bypass-duct
    "sensor_7": "P30[psi]", # Total pressure at HPC outlet
    "sensor_8": "Nf[rpm]", # Physical fan speed
    "sensor_9": "Nc[rpm]", # Physical core speed
    "sensor_10": "epr[-]", # Engine pressure ratio (P50/P2)
    "sensor_11": "phi[pph/psi]", # Ratio of fuel flow to Ps30, (not pps/psi, but pph/psi)
    "sensor_12": "Ps30[psi]", # Static pressure at HPC outlet
    "sensor_13": "NRf[rpm]", # Corrected fan speed
    "sensor_14": "NRc[rpm]", # Corrected core speed
    "sensor_15": "BPR[-]", # Bypass Ratio
    "sensor_16": "farB[-]", # Burner fuel-air ratio
    "sensor_17": "htBleed[]",# Bleed Enthalpy
    "sensor_18": "Nf_dmd[rpm]", # Demanded fan speed
    "sensor_19": "PCNfR_dmd[Pct]", # Demanded corrected fan speed
    "sensor_20": "W31[lbm/s]", # HPT coolant bleed
    "sensor_21": "W32[lbm/s]", # LPT coolant bleed
}


train_df = train_df.rename(columns=rename_dict)
test_df  = test_df.rename(columns=rename_dict)

def clean_round(series, ndigits=1, eps=1e-6):
    """Round values and force -0.0 to 0.0"""
    rounded = series.round(ndigits)
    rounded[rounded.abs() < eps] = 0
    return rounded

train_df['condition'] = (
    clean_round(train_df["Alt[kft]"], 0).astype(str) + '_' +
    clean_round(train_df["Mn[-]"], 1).astype(str) + '_' +
    clean_round(train_df["TLA[deg]"], 0).astype(str)
)

train_df.head()

In [ ]:
train_df["P50[psi]"] = train_df["epr[-]"]*train_df["P2[psi]"]

train_df["Fan.PR[-]"] = train_df["P15[psi]"]/train_df["P2[psi]"]
train_df["LPC.TR[-]"] = train_df["T24[R]"]/train_df["T2[R]"] # Fan core + LPC
train_df["HPC.TR[-]"] = train_df["T30[R]"]/train_df["T24[R]"]

train_df["OPR[-]"] = train_df["P30[psi]"]/train_df["P2[psi]"]

train_df["Wf[pph]"] = train_df["phi[pph/psi]"]*train_df["Ps30[psi]"]
train_df["Wa36[lbm/s]"] = train_df["Wf[pph]"]/3600.0 / train_df["farB[-]"]
train_df["W24[lbm/s]"] = train_df["Wa36[lbm/s]"] + train_df["W31[lbm/s]"] + train_df["W32[lbm/s]"] # core. htBleed ? 
train_df["W15[lbm/s]"] = train_df["W24[lbm/s]"]*train_df["BPR[-]"] # bypass
train_df["W2[lbm/s]"] = train_df["W15[lbm/s]"] + train_df["W24[lbm/s]"] # overall

train_df["WfP3C[pph/psi]"] = train_df["phi[pph/psi]"]/np.sqrt(train_df["T2[R]"]/518.67)

train_df.head()


### 컬럼 설명 (Data Dictionary)

| 컬럼명 | 설명 | 타입 |
|---|---|---|
| **unit_number** | 엔진 고유 식별자 (Unit Number) | int64 |
| **time_in_cycles** | 운전 사이클 (Time in Cycles) | int64 |
| **Alt[kft]** | 고도 (Altitude) | float64 |
| **Mn[-]** | 마하 수 (Mach Number) | float64 |
| **TLA[deg]** | 스로틀 레버 각도 (Thrust Lever Angle) | float64 |
| **T2[R]** | 팬 입구 전온도 (Total temperature at fan inlet) | float64 |
| **T24[R]** | LPC 출구 전온도 (Total temperature at LPC outlet) | float64 |
| **T30[R]** | HPC 출구 전온도 (Total temperature at HPC outlet) | float64 |
| **T50[R]** | LPT 출구 전온도 (Total temperature at LPT outlet) | float64 |
| **P2[psi]** | 팬 입구 압력 (Pressure at fan inlet) | float64 |
| **P15[psi]** | 바이패스 덕트 전압력 (Total pressure in bypass-duct) | float64 |
| **P30[psi]** | HPC 출구 전압력 (Total pressure at HPC outlet) | float64 |
| **Nf[rpm]** | 물리적 팬 속도 (Physical fan speed) | float64 |
| **Nc[rpm]** | 물리적 코어 속도 (Physical core speed) | float64 |
| **epr[-]** | 엔진 압력비 (Engine pressure ratio) | float64 |
| **phi[pph/psi]** | 연료 유량 대 Ps30 비율 (Ratio of fuel flow to Ps30) | float64 |
| **Ps30[psi]** | HPC 출구 정압 (Static pressure at HPC outlet) | float64 |
| **NRf[rpm]** | 보정된 팬 속도 (Corrected fan speed) | float64 |
| **NRc[rpm]** | 보정된 코어 속도 (Corrected core speed) | float64 |
| **BPR[-]** | 바이패스 비 (Bypass Ratio) | float64 |
| **farB[-]** | 연소기 연료-공기 비 (Burner fuel-air ratio) | float64 |
| **htBleed[]** | 블리드 엔탈피 (Bleed Enthalpy) | int64 |
| **Nf_dmd[rpm]** | 요구 팬 속도 (Demanded fan speed) | int64 |
| **PCNfR_dmd[Pct]** | 요구 보정 팬 속도 (Demanded corrected fan speed) | float64 |
| **W31[lbm/s]** | HPT 냉각 블리드 (HPT coolant bleed) | float64 |
| **W32[lbm/s]** | LPT 냉각 블리드 (LPT coolant bleed) | float64 |
| **condition** | 운전 조건 문자열 (Operational Condition String) | object |
| **P50[psi]** | LPT 출구 전압력 (Total pressure at LPT outlet) | float64 |
| **Fan.PR[-]** | 팬 압력비 (Fan pressure ratio) | float64 |
| **LPC.TR[-]** | LPC 전온도비 (LPC total temperature ratio) | float64 |
| **HPC.TR[-]** | HPC 전온도비 (HPC total temperature ratio) | float64 |
| **OPR[-]** | 전체 압력비 (Overall pressure ratio) | float64 |
| **Wf[pph]** | 연료 유량 (Fuel flow) | float64 |
| **Wa36[lbm/s]** | 코어 공기 유량 (Core airflow) | float64 |
| **W24[lbm/s]** | LPC 공기 유량 (LPC airflow) | float64 |
| **W15[lbm/s]** | 바이패스 공기 유량 (Bypass airflow) | float64 |
| **W2[lbm/s]** | 팬 유입 공기 유량 (Fan inlet airflow) | float64 |
| **WfP3C[pph/psi]** | P3 압력 대비 연료 유량 (Fuel flow to P3 pressure ratio) | float64 |
| **RUL** | 엔진의 남은 수명 (Remaining Useful Life) | int64 |

In [ ]:
train_df.info()

In [ ]:
pd.set_option('display.max_rows', None)
summary_df = pd.DataFrame(train_df.dtypes, columns=['Data Type'])
summary_df = summary_df.reset_index()
summary_df = summary_df.rename(columns={'index': 'Column Name'})
summary_df['Non-Null Count'] = train_df.count().values
summary_df['Null Count'] = train_df.isnull().sum().values
summary_df['Null Ratio (%)'] = (train_df.isnull().sum().values / len(train_df)) * 100
summary_df


### 정규화 / 이상치 처리 / 노이즈 제거
* FD002는 FD001과 다르게 운전 조건이 단일 조건이 아니기 때문에 고도와 속도가 변한다.
1. 따라서 데이터를 여러 그룹(클러스터)로 나눠 (고도(Altitude), 마하수(Mach Number), 스로틀(TRA) 조합에 따라 총 6가지 운전 모드로 클러스터링)
2. 그룹별로 정규화를 하고 (Z-score) (고도나 속도 마다 온도 등 센서 수치가 다르기 때문에 이 차이를 없애기 위해 정규화) (trand 분리화)
3. z-score가 ±3을 벗어나는 이상치를 clipping 한다.
4. 정규화된 데이터를 다시 합쳐 모델에 넣는다.

* 노이즈 제거는 과정에서 컬럼이 매우 많이 파생되므로 EDA, feature selection 이후에 진행한다.

In [ ]:
# 노이즈 제거 함수 정의

from scipy.signal import savgol_filter

def add_advanced_features(df):
    """
    [기능]
    1. Savitzky-Golay 필터: 원본 센서의 노이즈를 매끄럽게 제거 (Technique B)
    2. EMA(지수이동평균): 최신 트렌드를 빠르게 반영 (Technique A)
    3. Rolling Window (5, 20): 기존의 단기/장기 트렌드 정보 유지
    """
    df_proc = df.copy()
    
    # 적용 대상: 'sensor'가 포함되거나 단위 '['가 붙은 컬럼
    # (주의: unit_number, time_in_cycles, RUL 등은 제외됨)
    target_cols = [c for c in df_proc.columns if ('sensor' in c or '[' in c) and c not in ['unit_number', 'time_in_cycles', 'RUL']]
    
    print(f"   👉 Advanced Feature 적용 대상: {len(target_cols)}개 컬럼")
    
    for col in target_cols:
        grouped = df_proc.groupby('unit_number')[col]
        
        # [Technique B] Savitzky-Golay 필터 (노이즈 제거)
        # 데이터가 충분할 때만 적용 (window 11)
        try:
            df_proc[col] = grouped.transform(
                lambda x: savgol_filter(x, window_length=11, polyorder=2) if len(x) > 11 else x
            )
        except:
            pass 

        # [Technique A] EMA (지수 이동 평균, span=20)
        # 늦은 예측 방지에 탁월함
        df_proc[f"{col}_ema_20"] = grouped.ewm(span=20).mean().reset_index(0, drop=True)
        
        # [기존 기능] Rolling Window 5 & 20
        df_proc[f"{col}_mean_5"] = grouped.rolling(window=5, min_periods=1).mean().reset_index(0, drop=True)
        df_proc[f"{col}_std_5"]  = grouped.rolling(window=5, min_periods=1).std().reset_index(0, drop=True)
        df_proc[f"{col}_mean_20"] = grouped.rolling(window=20, min_periods=1).mean().reset_index(0, drop=True)
        df_proc[f"{col}_std_20"] = grouped.rolling(window=20, min_periods=1).std().reset_index(0, drop=True)
        
    return df_proc

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def apply_clustering_and_normalization(df):
    df_norm = df.copy()
    
    # ---------------------------------------------------------
    # 1. 클러스터링 (운전 조건 그룹화)
    # ---------------------------------------------------------
    # FD002의 핵심: 고도, 마하수, 스로틀 기준으로 운전 조건(Condition) 나누기
    kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
    df_norm['cluster'] = kmeans.fit_predict(df_norm[['Alt[kft]', 'Mn[-]', 'TLA[deg]']])
    
    print("✅ 클러스터링 완료: 6개의 운전 조건 그룹 생성됨")

    # ---------------------------------------------------------
    # 2. 정규화 (Cluster-based Z-Score)
    # ---------------------------------------------------------
    # 각 클러스터(운전 조건) 별로 평균과 표준편차를 구해서 정규화
    # (이렇게 해야 고도 0일 때와 고도 40k일 때의 센서값을 공정하게 비교 가능)
    
    sensor_cols = [c for c in df_norm.columns if 'sensor' in c or '[' in c]
    # 제외할 컬럼들 (설정값 등)
    exclude = ['unit_number', 'time_in_cycles', 'RUL', 'cluster', 'Alt[kft]', 'Mn[-]', 'TLA[deg]']
    target_sensors = [c for c in sensor_cols if c not in exclude]

    print(f"✅ 정규화 대상 센서: {len(target_sensors)}개")

    for col in target_sensors:
        # 클러스터별로 그룹지어 정규화 (표준화)
        # transform 함수를 써서 각 그룹의 mean/std를 적용
        df_norm[col] = df_norm.groupby('cluster')[col].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-6) # 0나눗셈 방지
        )
        
    # ---------------------------------------------------------
    # 3. 이상치 처리 (Outlier Clipping)
    # ---------------------------------------------------------
    # Z-Score가 ±3을 넘어가면 ±3으로 꾹 눌러줌 (튀는 값 억제)
    for col in target_sensors:
        df_norm[col] = df_norm[col].clip(-3, 3)
        
    print("✅ 정규화 및 이상치 처리(Clipping) 완료")
    
    return df_norm

# 실행
train_norm = apply_clustering_and_normalization(train_df)
print(f"결과 데이터셋: train_norm (컬럼 수: {train_norm.shape[1]}) - 파생변수 없음!")

In [ ]:
# 수정 포인트: train_df -> train_norm 으로 변경

# 1. 각 클러스터별 설정(Setting) 값의 평균 확인
# (train_norm에는 cluster 컬럼과 원본 설정값이 모두 들어있습니다)
cluster_profile = train_norm.groupby('cluster')[['Alt[kft]', 'Mn[-]', 'TLA[deg]']].mean()
cluster_count = train_norm['cluster'].value_counts()

# 2. 보기 좋게 정렬 및 병합 (개수 포함)
cluster_profile['count'] = cluster_count

# 고도(Alt) 순서대로 정렬해서 출력 (물리적 의미 파악 용이)
print("📊 Cluster Profile (Operating Conditions):")
print(cluster_profile.sort_values('Alt[kft]'))

* 클러스터링 시각화

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ---------------------------------------------------------
# 1. 6개 운전 조건(Centroid) 정의
# ---------------------------------------------------------
# 사용자님이 제공하신 테이블 데이터 (Cluster ID 기준)
data = {
    'Cluster': [3, 4, 1, 2, 5, 0],
    'Alt': [0.0, 10.0, 20.0, 25.0, 35.0, 42.0],
    'Mn':  [0.0, 0.25, 0.70, 0.62, 0.84, 0.84],
    'TLA': [100.0, 100.0, 100.0, 60.0, 100.0, 100.0],
    'Desc': ["Sea Level", "Low Alt 1", "Low Alt 2", "Mid Alt (Cruise)", "High Alt 1", "High Alt 2"]
}
df_centroids = pd.DataFrame(data).sort_values('Alt')

# ---------------------------------------------------------
# 2. 경계벽(Wall) 위치 계산
# ---------------------------------------------------------
alts = df_centroids['Alt'].values
wall_positions = [(alts[i] + alts[i+1])/2 for i in range(len(alts)-1)]
boundaries = [-5] + wall_positions + [45]

# Y축(Mn), Z축(TLA) 범위 설정
y_min, y_max = -0.1, 1.0
z_min, z_max = 0, 110

# ---------------------------------------------------------
# 3. 3D 공간 분할 시각화
# ---------------------------------------------------------
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# (1) 경계벽(Wall) 그리기 (색상 변경: Skyblue)
# X축(Alt)을 기준으로 수직 벽을 세웁니다.
colors = plt.cm.viridis(np.linspace(0, 1, len(df_centroids)))

for i, x_pos in enumerate(wall_positions):
    verts = [
        [(x_pos, y_min, z_min), (x_pos, y_max, z_min), 
         (x_pos, y_max, z_max), (x_pos, y_min, z_max)]
    ]
    
    # 🎨 [수정됨] 잘렸던 부분 수정 완료 (linewidth=0.5)
    wall = Poly3DCollection(verts, alpha=0.15, facecolor='deepskyblue', edgecolor='royalblue', linewidth=0.5)
    ax.add_collection3d(wall)
    
    # 벽 라벨
    ax.text(x_pos, y_max, z_max + 5, f"Alt={x_pos:.1f}", color='blue', fontsize=9, ha='center')

# (2) 중심점(Centroid) 및 영역 표시
for i, row in df_centroids.iterrows():
    # 중심점
    ax.scatter(row['Alt'], row['Mn'], row['TLA'], 
               color=colors[i], s=200, edgecolor='k', alpha=1.0, label=f"Cluster {int(row['Cluster'])}")
    
    # 텍스트 라벨
    ax.text(row['Alt'], row['Mn'], row['TLA'] + 5, 
            f"Cluster {int(row['Cluster'])}\n({row['Desc']})", 
            color='black', fontsize=10, fontweight='bold', ha='center')
    
    # 바닥 투영선
    ax.plot([row['Alt'], row['Alt']], [row['Mn'], row['Mn']], [0, row['TLA']], 
            color=colors[i], linestyle='--', alpha=0.5)

# ---------------------------------------------------------
# 4. 그래프 꾸미기
# ---------------------------------------------------------
ax.set_xlabel('Altitude [kft]', fontsize=12, labelpad=10)
ax.set_ylabel('Mach Number [-]', fontsize=12, labelpad=10)
ax.set_zlabel('Throttle [deg]', fontsize=12, labelpad=10)
ax.set_title("6 Operating Zones (Blue Wall View)", fontsize=16, fontweight='bold', pad=20)

ax.set_xlim(-5, 50)
ax.set_ylim(y_min, y_max)
ax.set_zlim(0, 120)

ax.view_init(elev=20, azim=-70)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5. [복구 완료] 구간 정의표 출력
# ---------------------------------------------------------
print("\n [Operating Zones & Boundaries Table]")
print("=" * 75)
print(f"{'Zone':<6} | {'Cluster':<8} | {'Altitude Range [kft]':<25} | {'Description'}")
print("-" * 75)

for i in range(len(boundaries)-1):
    low = boundaries[i]
    high = boundaries[i+1]
    cluster_id = df_centroids.iloc[i]['Cluster']
    desc = df_centroids.iloc[i]['Desc']
    
    range_str = f"{low:.1f} < Alt < {high:.1f}"
    if i == 0: range_str = f"Alt < {high:.1f} (Ground)"
    if i == len(boundaries)-2: range_str = f"Alt > {low:.1f} (High Alt)"
    
    print(f" {i+1:<5} | {int(cluster_id):<8} | {range_str:<25} | {desc}")
print("=" * 75)

In [ ]:
def add_rul(df):
    """각 엔진별 RUL(잔여 수명) 계산"""
    max_cycles = df.groupby('unit_number')['time_in_cycles'].max()
    df = df.merge(max_cycles.to_frame(name='max_cycle'), 
                  left_on='unit_number', right_index=True)
    df['RUL'] = df['max_cycle'] - df['time_in_cycles']
    df.drop('max_cycle', axis=1, inplace=True)
    return df
# train_df에 RUL 추가
train_df = add_rul(train_df)
print(f"✅ RUL 컬럼 추가 완료!")
print(train_df[['unit_number', 'time_in_cycles', 'RUL']].head(10))

* 작업 전후 비교 전에 추세가 확실한 컬럼 3개 찾기

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# 🚨 [Fix] 대상 데이터프레임 변경 (train_final -> train_norm)
# ---------------------------------------------------------
# 현재 노이즈 제거(train_final 생성) 전이므로, 정규화된 'train_norm'을 사용합니다.

# RUL 컬럼이 없다면 이식
if 'RUL' not in train_norm.columns:
    train_norm['RUL'] = train_df['RUL']

print(f"✅ train_norm 데이터 준비 완료! (컬럼 수: {train_norm.shape[1]})")

# ---------------------------------------------------------
# 1. 어떤 센서가 RUL과 가장 관련이 깊을까? (상관관계 분석)
# ---------------------------------------------------------
# 분석 대상: 정규화된 'train_norm'
# 클러스터링 관련 컬럼 제외하고 순수 센서만 선택
features = [c for c in train_norm.columns if ('sensor' in c or '[' in c) 
            and c not in ['RUL', 'unit_number', 'time_in_cycles', 'cluster', 'Alt[kft]', 'Mn[-]', 'TLA[deg]']]

# 상관관계 계산
correlations = train_norm[features + ['RUL']].corr()['RUL'].drop('RUL')
top_features = correlations.abs().sort_values(ascending=False).head(3)

print("\n🏆 RUL과 가장 강력한 추세를 가진 Top 3 센서 (Normalized):")
print(top_features)

# ---------------------------------------------------------
# 2. Top 1 센서의 "진짜 추세" 시각화
# ---------------------------------------------------------
if len(top_features) > 0:
    best_sensor = top_features.index[0] # 1등 센서 이름

    # 데이터 샘플링 (속도 향상)
    sample_df = train_norm.sample(frac=0.1, random_state=42)

    plt.figure(figsize=(12, 6))
    
    # 산점도 그리기
    plt.scatter(sample_df['RUL'], sample_df[best_sensor], 
                alpha=0.1, s=10, c='red', label='Data Points')

    # 추세선 (Trend Line) 추가
    sns.regplot(data=sample_df, x='RUL', y=best_sensor, scatter=False, 
                color='black', line_kws={'linestyle':'--', 'linewidth':2})

    plt.gca().invert_xaxis() # X축 반전: 오른쪽(0)이 고장 시점
    plt.title(f"Best Trend Feature: {best_sensor} vs RUL (Normalized)", fontsize=15, fontweight='bold')
    plt.xlabel("Remaining Useful Life (RUL) -> Decreasing")
    plt.ylabel(f"Normalized Value (Z-Score of {best_sensor})")
    plt.axhline(0, color='blue', linestyle=':', alpha=0.5, label='Mean (0)') # 평균선
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("❌ 상관관계를 계산할 센서 컬럼을 찾지 못했습니다.")

* 정규화 / 이상치 처리 / 노이즈 제거 작업 전후 비교

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ---------------------------------------------------------
# 1. [단일 엔진] 시계열 변화 비교 (Raw vs Normalized)
# ---------------------------------------------------------
def plot_trajectory_comparison_step5(unit_id, sensor_name):
    # Step 5 시점: 아직 노이즈 제거(Step 10) 전이므로 2개만 비교합니다.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # (1) Raw Data (train_df)
    if sensor_name in train_df.columns:
        axes[0].plot(train_df[train_df['unit_number']==unit_id][sensor_name], 
                     'b-', alpha=0.6, linewidth=1)
        axes[0].set_title(f"1. Raw Data\n(Condition Mixed)", fontsize=12, fontweight='bold')
    else:
        axes[0].text(0.5, 0.5, "Column Not Found", ha='center')
        axes[0].set_title(f"1. Raw Data", fontsize=12)

    axes[0].set_ylabel("Sensor Value")
    axes[0].set_xlabel("Time (Cycles)")
    axes[0].grid(True, alpha=0.3)
    
    # (2) Normalized Data (train_norm)
    # 정규화 및 이상치 처리(Clipping)가 된 데이터
    if 'train_norm' in globals() and sensor_name in train_norm.columns:
        axes[1].plot(train_norm[train_norm['unit_number']==unit_id][sensor_name], 
                     'g-', alpha=0.6, linewidth=1)
        axes[1].set_title(f"2. Normalized & Clipped\n(Condition Removed)", fontsize=12, fontweight='bold')
    else:
        axes[1].text(0.5, 0.5, "Not in train_norm", ha='center')
        axes[1].set_title(f"2. Normalized", fontsize=12)

    axes[1].set_ylabel("Z-Score (Clipped)")
    axes[1].set_xlabel("Time (Cycles)")
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(f"Preprocessing Effect: {sensor_name} (Unit {unit_id})", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# 2. [전체 데이터] Global Trend 비교 (Raw vs Normalized)
# ---------------------------------------------------------
def plot_global_trend_step5(sensor_name):
    # 데이터 샘플링 (10%)
    sample_raw = train_df.sample(frac=0.1, random_state=42)
    
    # train_norm에서도 같은 인덱스로 샘플링
    if 'train_norm' in globals():
        sample_norm = train_norm.loc[sample_raw.index]
    else:
        print("⚠️ train_norm 데이터가 없습니다.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # (1) Raw Data vs RUL
    if sensor_name in sample_raw.columns:
        axes[0].scatter(sample_raw['RUL'], sample_raw[sensor_name], alpha=0.1, s=10, c='blue')
    axes[0].set_title(f"Raw Data vs RUL", fontsize=12)
    axes[0].set_xlabel("RUL")
    axes[0].set_ylabel("Raw Value")
    axes[0].invert_xaxis()
    axes[0].grid(True, alpha=0.3)
    
    # (2) Normalized Data vs RUL
    if sensor_name in sample_norm.columns:
        axes[1].scatter(sample_raw['RUL'], sample_norm[sensor_name], alpha=0.1, s=10, c='green')
        axes[1].set_title(f"Normalized Data vs RUL\n(Condition Removed)", fontsize=12, fontweight='bold')
    else:
        axes[1].text(0.5, 0.5, "Column Not Found", ha='center')
        
    axes[1].set_xlabel("RUL")
    axes[1].set_ylabel("Z-Score")
    axes[1].invert_xaxis()
    axes[1].grid(True, alpha=0.3)
    
    plt.show()

# ---------------------------------------------------------
# 🔥 실행 (Step 5 시점)
# ---------------------------------------------------------
# RUL 컬럼이 train_norm에 없으면 이식 (시각화용)
if 'train_norm' in globals() and 'RUL' not in train_norm.columns:
    train_norm['RUL'] = train_df['RUL']

# 시각화할 주요 센서 (우선순위 센서 위주)
target_sensors = ['WfP3C[pph/psi]', 'phi[pph/psi]', 'T50[R]', 'P30[psi]']

# 컬럼 있는지 확인 후 실행
for sensor in target_sensors:
    if sensor in train_df.columns:
        print(f"\n🔍 Comparing Raw vs Normalized: {sensor}")
        plot_trajectory_comparison_step5(unit_id=2, sensor_name=sensor)
    else:
        print(f"⚠️ {sensor} 컬럼이 데이터에 없습니다.")

# 전체 트렌드 확인 (1번 타자)
if target_sensors[0] in train_df.columns:
    print(f"\n🌍 Global Trend Check ({target_sensors[0]})")
    plot_global_trend_step5(target_sensors[0])

* 왼쪽 그래프의 6개 선은 6가지 운전 조건을 의미 (운전조건의 영향이 강해 고장에 의한 추세(Trend)가 안 보인다.)
* 선들이 사라지고 추세가 보인다.

### RUL Clipping

In [ ]:
# 보통 RUL이 125보다 크면 모두 125로 고정 (Piecewise Linear RUL)
# 이유: 초기 건강한 구간에서의 불필요한 학습 혼란 방지
# GridSearch 를 이용해 [100, 110, 120, 125, 130, 140, 150] 중에 최적 클립 구간 찾기


from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error

# RUL 컬럼 생성 
max_cycle = train_df.groupby('unit_number')['time_in_cycles'].transform('max')
train_df['RUL'] = max_cycle - train_df['time_in_cycles']
print("RUL 컬럼 생성 완료!")

# ---------------------------------------------------------
# [Step 1] Grid Search로 최적의 RUL Clipping 포인트 탐색
# ---------------------------------------------------------
print("🔎 최적의 RUL 상한선(Clipping Limit) 탐색을 시작합니다...")

# 1. 탐색할 후보군 설정 (100 ~ 140 사이)
candidate_limits = [100, 110, 120, 125, 130, 140, 150]
best_rmse = float('inf')
best_limit = 125 # 기본값 (혹시 탐색 실패 시 안전장치)

# 문자열(object) 컬럼은 제외하고, 오직 숫자형(number) 컬럼만 선택합니다.
numeric_cols = train_df.select_dtypes(include=[np.number]).columns

# 2. 실험을 위한 임시 데이터 준비
# (현재 train_df에서 모델링에 사용할 컬럼만 뽑습니다)
features = [c for c in numeric_cols if c not in ['unit_number', 'time_in_cycles', 'RUL']]
groups = train_df['unit_number']

# 3. 반복문으로 후보군 테스트
for limit in candidate_limits:
    # (1) RUL Clipping 적용 (임시)
    # 원본 train_df는 건드리지 않고, y_target만 임시로 만듭니다.
    y_candidate = train_df['RUL'].clip(upper=limit)
    X_candidate = train_df[features]
    
    # (2) 데이터 분리 (엔진 단위 분할)
    gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
    train_idx, val_idx = next(gss.split(X_candidate, y_candidate, groups))
    
    X_tr, X_val = X_candidate.iloc[train_idx], X_candidate.iloc[val_idx]
    y_tr, y_val = y_candidate.iloc[train_idx], y_candidate.iloc[val_idx]
    
    # (3) 가벼운 모델로 빠르게 성능 측정 (Linear Regression)
    model = LinearRegression()
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    
    # (4) RMSE 계산
    mse = mean_squared_error(y_val, pred)
    rmse = np.sqrt(mse)
    
    print(f"   👉 테스트 Limit: {limit} -> RMSE: {rmse:.4f}")
    
    # (5) 최고 기록 갱신 여부 체크
    if rmse < best_rmse:
        best_rmse = rmse
        best_limit = limit

print(f"\n🏆 [탐색 결과] 최적의 RUL Limit은 '{best_limit}'입니다. (RMSE: {best_rmse:.4f})")


# ---------------------------------------------------------
# [Step 2] 찾은 최적값(best_limit)을 실제 데이터에 적용
# ---------------------------------------------------------
train_df['RUL'] = train_df['RUL'].clip(upper=best_limit)

print(f"✅ RUL Clipping 완료! (최대값 {best_limit}로 제한 적용됨)")
print(train_df['RUL'].describe())

# EDA

## unit 별 최대 수명 분석

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 데이터 준비 (이미 하신 부분)
index_names = ['unit_number', 'time_in_cycles']
max_time_cycles = train_df[index_names].groupby('unit_number').max()
max_time_cycles = max_time_cycles.reset_index() # unit_number를 컬럼으로 뺌

# 통계치 계산
mean_life = max_time_cycles['time_in_cycles'].mean()
max_life = max_time_cycles['time_in_cycles'].max()
min_life = max_time_cycles['time_in_cycles'].min()

# ---------------------------------------------------------
# 시각화: 3가지 관점 (산점도, 히스토그램, 정렬 분포)
# ---------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid') # 깔끔한 스타일
fig = plt.figure(figsize=(20, 12))

# 1. 산점도 (Scatter Plot) - 모든 유닛을 펼쳐서 보기
# X축: 유닛 번호, Y축: 수명
ax1 = plt.subplot(2, 2, (1, 2)) # 윗부분 전체 사용
sns.scatterplot(data=max_time_cycles, x='unit_number', y='time_in_cycles', 
                color='royalblue', alpha=0.7, s=60, edgecolor='w', ax=ax1)

# 평균선, 최대/최소 표시
ax1.axhline(mean_life, color='red', linestyle='--', label=f'Mean Life: {mean_life:.1f}')
ax1.text(0, mean_life + 5, f' Mean: {mean_life:.1f}', color='red', fontweight='bold')

# 가장 오래 산 엔진과 빨리 죽은 엔진 표시
ax1.annotate(f'Max: {max_life} (Unit {max_time_cycles.iloc[max_time_cycles["time_in_cycles"].idxmax()]["unit_number"]})',
             xy=(max_time_cycles.iloc[max_time_cycles["time_in_cycles"].idxmax()]["unit_number"], max_life),
             xytext=(10, -20), textcoords='offset points', arrowprops=dict(arrowstyle="->", color='black'))

ax1.annotate(f'Min: {min_life} (Unit {max_time_cycles.iloc[max_time_cycles["time_in_cycles"].idxmin()]["unit_number"]})',
             xy=(max_time_cycles.iloc[max_time_cycles["time_in_cycles"].idxmin()]["unit_number"], min_life),
             xytext=(10, 20), textcoords='offset points', arrowprops=dict(arrowstyle="->", color='black'))

ax1.set_title(f'Turbofan Engines LifeTime (Total {len(max_time_cycles)} Units)', fontsize=18, fontweight='bold')
ax1.set_xlabel('Unit Number', fontsize=14)
ax1.set_ylabel('Max Time Cycles', fontsize=14)
ax1.legend()


# 2. 히스토그램 (Histogram) - 수명 분포 확인
# 엔진들이 보통 얼마쯤 살다 죽는지 분포 확인
ax2 = plt.subplot(2, 2, 3)
sns.histplot(max_time_cycles['time_in_cycles'], bins=30, kde=True, color='green', ax=ax2)
ax2.axvline(mean_life, color='red', linestyle='--')
ax2.set_title('Distribution of LifeTimes', fontsize=16, fontweight='bold')
ax2.set_xlabel('Max Time Cycles', fontsize=12)


# 3. 정렬된 막대 그래프 (Sorted Bar/Area) - 순위 비교
# 수명이 짧은 순서대로 정렬해서 보여줌 (Survival Curve 형태)
sorted_life = max_time_cycles.sort_values('time_in_cycles').reset_index(drop=True)

ax3 = plt.subplot(2, 2, 4)
ax3.plot(sorted_life.index, sorted_life['time_in_cycles'], color='purple', linewidth=2)
ax3.fill_between(sorted_life.index, sorted_life['time_in_cycles'], color='purple', alpha=0.2)
ax3.set_title('Sorted LifeTime Curve (Min to Max)', fontsize=16, fontweight='bold')
ax3.set_xlabel('Rank (Sorted Units)', fontsize=12)
ax3.set_ylabel('Max Time Cycles', fontsize=12)
ax3.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 시계열 그래프, 산점도, 히트맵으로 feature, RUL 관계 분석

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

class FeatureAnalysisPlot:
    def __init__(self, dataframe):
        self.df = dataframe.copy()
        self.warnings = warnings
        
        # 1. 스타일 설정 (기존 유지: bmh)
        plt.style.use('bmh')
        
        # 2. RUL 컬럼 확인
        if 'RUL' not in self.df.columns:
            print("⚠️ 데이터프레임에 'RUL' 컬럼이 없습니다. 확인해주세요.")
            
        # 3. 분석할 센서 자동 선별
        exclude_cols = ['unit_number', 'time_in_cycles', 'RUL', 'max_cycle', 'cluster', 'condition', 'Mode_ID']
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        
        self.sensors = [c for c in numeric_cols if c not in exclude_cols and ('sensor' in c or '[' in c)]
        self.sensors.sort() 
        
        self.cycles = self.df["time_in_cycles"]

    # ---------------------------------------------------------
    # 1. 시계열 그래프 (기존 유지)
    # ---------------------------------------------------------
    def plot_time_series(self, unit_id):
        single_unit = self.df[self.df["unit_number"] == unit_id]
        
        if len(single_unit) == 0:
            print(f"⚠️ Unit {unit_id} 데이터가 없습니다.")
            return

        n_sensors = len(self.sensors)
        cols = 2
        rows = math.ceil(n_sensors / cols) 
        
        colors = iter(plt.cm.plasma(np.linspace(0, 1, n_sensors)))
        
        plt.figure(figsize=(8, 2 * rows)) 
        
        for i, sensor in enumerate(self.sensors):
            c = next(colors)
            plt.subplot(rows, cols, i+1)
            
            plt.plot(single_unit['time_in_cycles'], single_unit[sensor], 
                     c=c, label=sensor, linewidth=1.5)
            
            plt.title(f"{sensor} (Normalized)", fontsize=9, fontweight='bold')
            plt.xlabel("Cycles", fontsize=8)
            plt.ylabel("Z-Score", fontsize=8)
            plt.tick_params(labelsize=7)
            plt.grid(True, alpha=0.3)
            
        plt.tight_layout()
        plt.show()

    # ---------------------------------------------------------
    # 2. 산점도 (✅ 수정됨: X축=RUL, Y축=Sensor)
    # ---------------------------------------------------------
    def plot_scatter(self):
        n_sensors = len(self.sensors)
        cols = 3
        rows = math.ceil(n_sensors / cols)
        
        # 데이터 샘플링 (10%)
        sample_df = self.df.sample(frac=0.1, random_state=42)
        
        with self.warnings.catch_warnings(record=True):
            plt.figure(figsize=(10, 2.5 * rows)) # 사이즈 유지
            
            for i, sensor in enumerate(self.sensors):
                plt.subplot(rows, cols, i+1)
                
                # [변경] X축을 RUL로 설정
                scatter = plt.scatter(
                    x=sample_df["RUL"],      # X축: RUL
                    y=sample_df[sensor],     # Y축: 센서값
                    s=1,              
                    alpha=0.3,        
                    c=sample_df["time_in_cycles"],    
                    cmap=plt.get_cmap("plasma") 
                )
                
                plt.xlabel("RUL", fontsize=8)      # 라벨 변경
                plt.ylabel(sensor, fontsize=8)     # 라벨 변경
                plt.title(f"{sensor} vs RUL", fontsize=9)
                plt.gca().invert_xaxis()           # [추가] RUL 감소 방향으로 뒤집기
                plt.tick_params(labelsize=7)
                plt.grid(True, alpha=0.3)
                
            plt.tight_layout()
            plt.show()
    
    # ---------------------------------------------------------
    # 3. 히트맵 (기존 유지)
    # ---------------------------------------------------------
    def plot_heatmap(self):
        cols_to_corr = self.sensors + ['RUL']
        cols_to_corr = [c for c in cols_to_corr if c in self.df.columns]
        
        corr_mat = self.df[cols_to_corr].corr()
        
        sorted_idx = corr_mat['RUL'].abs().sort_values(ascending=False).index
        sorted_corr = corr_mat.loc[sorted_idx, sorted_idx]
        
        plt.figure(figsize=(16, 12)) 
        
        sns.heatmap(sorted_corr, 
                    cmap="coolwarm", center=0, annot=True, fmt='.2f', 
                    linewidths=.5, annot_kws={"size": 10},
                    cbar_kws={"shrink": .8})
        
        plt.title("Feature Correlation with RUL (Normalized Data)", fontsize=16, fontweight='bold')
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(fontsize=10)
        plt.show()

# ---------------------------------------------------------
# 🔥 실행 코드 (Step 8)
# ---------------------------------------------------------
# 1. train_norm에 RUL 확인
if 'RUL' not in train_norm.columns:
    train_norm['RUL'] = train_df['RUL']

print(f"📊 [Step 8] Feature Analysis with train_norm (Shape: {train_norm.shape})")

# 2. 객체 생성 및 시각화
analysis_plot = FeatureAnalysisPlot(train_norm)

print("\n📈 1. Time Series (Unit 1) - 정규화된 데이터의 추세 확인")
analysis_plot.plot_time_series(unit_id=1)

print("\n🌌 2. Scatter Plots - RUL과의 관계 확인 (X축: RUL로 변경됨)")
analysis_plot.plot_scatter()

print("\n🔥 3. Heatmap - 상관계수 확인 (Feature Selection 기준)")
analysis_plot.plot_heatmap()

### 산점도에서 선형과 흩뿌려진 형태가 혼재하는 것을 확인 클러스터 차이인지 확인하기 위해 클러스터 별 색상 구현

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

# ---------------------------------------------------------
# [Step 8-1] 모든 센서 심화 진단 (Compact Style)
# ---------------------------------------------------------
def check_all_sensors_deep_dive_compact(df):
    # 1. 분석할 센서 컬럼 자동 추출
    exclude_cols = ['unit_number', 'time_in_cycles', 'RUL', 'max_cycle', 'cluster', 'condition', 'Mode_ID']
    all_cols = df.columns
    sensor_cols = [c for c in all_cols if c not in exclude_cols and ('sensor' in c or '[' in c)]
    sensor_cols.sort()
    
    n_sensors = len(sensor_cols)
    print(f"🔍 총 {n_sensors}개의 센서를 분석합니다. (Cluster별 색상 구분)\n")

    # 샘플링 (속도 향상 및 시각적 겹침 방지)
    sample_df = df.sample(frac=0.2, random_state=42)
    
    # 2. Subplot 그리드 설정 (Step 8 스타일)
    cols = 3  # 한 줄에 3개씩
    rows = math.ceil(n_sensors / cols)
    
    # 그래프 전체 크기 설정 (개별 그래프가 작게 나오도록 조절)
    plt.figure(figsize=(15, 3.5 * rows))
    
    # 3. 모든 센서 반복 출력
    for i, sensor_name in enumerate(sensor_cols):
        plt.subplot(rows, cols, i+1)
        
        # 산점도 그리기
        # palette='tab10': 색상 구분이 뚜렷한 원색 계열 (파랑, 주황, 초록, 빨강...)
        sns.scatterplot(data=sample_df, x='RUL', y=sensor_name, 
                        hue='cluster', palette='tab10',  
                        s=5, alpha=0.6, legend=False) # 범례는 너무 많아서 생략 (필요하면 마지막에만 추가)
        
        plt.title(f"{sensor_name}", fontsize=11, fontweight='bold')
        plt.xlabel("RUL", fontsize=9)
        plt.ylabel("Normalized", fontsize=9)
        plt.gca().invert_xaxis() # RUL 감소 방향
        plt.grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.show()

    # (참고용) 범례만 따로 하나 출력
    print("\n🎨 [범례 참고] Cluster Colors (Tab10 Palette):")
    plt.figure(figsize=(6, 1))
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10) 
               for c in sns.color_palette('tab10', 6)]
    plt.legend(handles, [f'Cluster {i}' for i in range(6)], loc='center', ncol=6)
    plt.axis('off')
    plt.show()

# ---------------------------------------------------------
# 🔥 실행
# ---------------------------------------------------------
if 'train_norm' in globals():
    check_all_sensors_deep_dive_compact(train_norm)
else:
    print("⚠️ 'train_norm' 변수가 없습니다.")

## 결과

1. 산점도에서 산재해있지만 RUL과 다 같은 방향의 추세(trend)를 가지는 케이스
2. 산점도에서 산재해있지만 RUL과 클러스터 별로 감소,증가 두가지의 추세(trend)를 가지는 케이스
3. 직선으로 상수형 데이터를 보여주는 케이스
4. 산재형과 상수형이 혼재하는 케이스

위 4가지 케이스가 있고   
3번은 FD_001과 같이 삭제   
4번은 상수형이 모두 cluster 1인것을 보아 해당 운전 조건에서 측정이 안되는 상태 였던걸로 추측    
따라서 cluster의 차이가 있다는걸 모델에 알려주기 위해 'cluster' 컬럼은 유지 (단, 크기가 아니라 클러스터 구분용이니까 one-hot encoding 사용)

Featureselection

test data, 시각화 데이터 생성

모델링

결과 비교 및 분석